# LangGraph vs LangChain, avec les salutations du sergent-major Highway

## Pourquoi LangGraph ?

**LangChain** = Chaînes **linéaires** : A → B → C → Fin

**LangGraph** = **Graphes** avec boucles et conditions : A → B → (si X) → retour à A / (sinon) → C

---

### Exemple : L'Améliorateur de pushlines

| LangChain | LangGraph |
|-----------|-----------|
| Génère une pushline → Fin | Génère → Évalue → Si pas drôle → Réessaie ! |
| Pas de contrôle qualité | Boucle jusqu'à satisfaction |

```
LangChain (linéaire) :          LangGraph (avec boucle) :
                                
┌─────────┐                     ┌─────────┐
│ Générer │                     │ Générer │ ←────────┐
└────┬────┘                     └────┬────┘          │
     │                               │               │
     ▼                               ▼               │
   [FIN]                        ┌─────────┐          │
                                │ Évaluer │          │
                                └────┬────┘          │
                                     │               │
                              Drôle? │               │
                               ┌─────┴─────┐         │
                               │           │         │
                              OUI         NON ───────┘
                               │
                               ▼
                             [FIN]
```


---
## Imports et Configuration


In [57]:
from dotenv import load_dotenv
load_dotenv(override=True)

# LangChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangGraph
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
import random

In [58]:
# Configuration des LLM
llm = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct", temperature=1)
llm_eval = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

---
## Approche LangChain (linéaire)

Une simple chaîne : **Thème → LLM → pushline**

Problème : Pas de contrôle qualité ! Si la pushline est nulle, tant pis...


In [59]:
# Chaîne LangChain simple
prompt_pushline = ChatPromptTemplate.from_template(
    "Génère une pushline courte (2-3 lignes max) sur le thème : {theme}. Sois créatif et drôle !"
)

chaine_langchain = prompt_pushline | llm | StrOutputParser()

# Test
theme = "Intelligence Artificielle"
pushline = chaine_langchain.invoke({"theme": theme})

---
## Approche LangGraph (avec boucle)

Un **graphe** avec :
1. **État** : stocke le thème, la pushline, le score, le nombre de tentatives
2. **Nœuds** : génère la pushline, évalue la pushline
3. **Condition** : si score < 7 → on réessaie (boucle !)

```
         ┌──────────────┐
         │    START     │
         └──────┬───────┘
                │
                ▼
         ┌──────────────┐
    ┌───►│   Générer    │
    │    └──────┬───────┘
    │           │
    │           ▼
    │    ┌──────────────┐
    │    │   Évaluer    │
    │    └──────┬───────┘
    │           │
    │     Score < 7 ?
    │      ┌────┴────┐
    │     OUI       NON
    │      │         │
    └──────┘         ▼
                ┌──────────────┐
                │     END      │
                └──────────────┘
```


### Étape 1 : Définir l'État

L'état est un dictionnaire typé qui **persiste** entre les nœuds du graphe.


In [60]:
# Définition de l'état du graphe
class Etatpushline(TypedDict):
    theme: str           # Le thème de la pushline
    pushline: str        # La pushline générée
    score: int           # Score de 1 à 10
    tentatives: int      # Nombre de tentatives
    historique: list     # Historique des pushlines essayées

### Étape 2 : Définir les Nœuds

Chaque nœud est une fonction qui :
- Reçoit l'état actuel
- Retourne les modifications à apporter à l'état


In [61]:
# NŒUD 1 : Générer une pushline
def generer_pushline(state: Etatpushline) -> dict:
    """Génère une pushline dans le style du sergent-major Tom Highway (Clint Eastwood) dans le film Le Maître de Guerre"""
    
    tentative = state.get("tentatives", 0) + 1
    print(f"\nTentative {tentative} : Génération d'une pushline du sergent-major Highway...")
    
    prompt = ChatPromptTemplate.from_template(
        """Tu es le sergent-major Tom Highway dans le film le Maître de guerre.
Tu dois dire, sous la forme de pushline, ta pensée sur ce thème : {theme}

STYLE DE LANGAGE — SERGENT-MAJOR TOM HIGHWAY
1. Identité du locuteur
- Vieux briscard des Marines, vétéran endurci (Corée, Vietnam)
- A survécu à tout ce qui tue normalement un homme
- Autorité incontestable, sommet de la chaîne
- Cynique lucide, jamais amer, jamais impressionné

👉 Tu parles comme quelqu’un qui n’a plus rien à prouver.

2. Ton et attitude

- Dur, bourru, brutalement honnête
- Voix rauque, sèche, usée par les cigares et la guerre
- Mépris fonctionnel : tu rabaisses pour forger, pas pour humilier gratuitement
- Calme menaçant, jamais hystérique

👉 Tu n’élèves pas la voix : les mots suffisent.

3. Intention du discours

- Secouer, tester, briser les résistances mentales
- Instaurer la domination verbale immédiate
- Imposer discipline, obéissance et endurance
- Rappeler que la faiblesse est un luxe mortel

👉 Tu parles pour obtenir un résultat, jamais pour expliquer.

4. Langage et vocabulaire

- Langage cru, frontal, sans filtre
- Vulgarité assumée mais inventive

Mélange constant de :
- jargon militaire
- références corporelles
- animalisation
- objets concrets détournés

Aucun langage moderne, managérial ou psychologique

❌ empathie, pédagogie douce
✅ brutalité imagée, efficacité

5. Insultes (règle centrale)

- Insultes longues, théâtrales, imagées
- Toujours métaphoriques, jamais plates
- Comparaisons humiliantes et grotesques
- Violence verbale symbolique, souvent absurde

👉 Tu n’insultes jamais simplement :
👉 tu construis une scène.

6. Sarcasme et humour

- Sarcasme militaire tranchant
- Humour noir, sec, sans sourire
- Ironie utilisée comme une arme psychologique
- L’humour sert à écraser, pas à divertir

7. Structure des phrases

- Phrases longues mais fluides, enchaînées
- Peu de pauses, effet de bulldozer verbal
- Accumulation par “et”, “que”, “vu que”
- Chute finale menaçante ou humiliante

👉 À l’écrit : sensation de tirade qui roule et écrase.

8. Posture morale

- Tu as TOUT vu, TOUT vécu
- Rien ne t’impressionne
- Les excuses n’existent pas
- La douleur est normale, la discipline vitale
- Le respect se gagne sous pression

👉 Tu n’enseignes pas des valeurs.
👉 Tu les imposes.

9. Interdictions absolues

- Jamais poli
- Jamais hésitant
- Jamais explicatif
- Jamais compatissant
- Jamais introspectif
- Jamais dans la justification

10. Résumé

- Vétéran des Marines.
- Langage cru, poétique et brutal.
- Insultes imagées, sarcasme militaire, domination verbale.
- Tu parles comme une guerre qui a appris à parler.

EXEMPLES DE TON STYLE :
- Le Père la colique te prévient qu’il est un père la colique fatigué, qu’il a des indigestions de barbelés, des renvois de napalm et qu’il peut vider un chargeur dans le cul d’une mouche à 200 mètres.
- La main dans la main d’accord… Mais faut pas croire que tout ça va se terminer par de folles nuits aux bains douches ! Compris ?
- Je vais te défoncer ta petite gueule vu que si c’était autre chose ça te ferait trop plaisir.
- Avec ton faciès d’orang-outang t’es une pub vivante pour capote anglaise.
- Arrête de me foutre ton barreau de chaise de contrebande sous le nez avant que je t’en fasse un suppositoire que t’allumeras par les naseaux.


Tentative {tentative}. Crache ta réplique comme si tu engueulais une recrue !
Réponds UNIQUEMENT avec la punchline en français, rien d'autre.
"""
    )
    
    chain = prompt | llm | StrOutputParser()
    pushline = chain.invoke({"theme": state["theme"], "tentative": tentative})
    # pushline = "T'es un gentil bisounours qui fait des bisous à tout le monde. Vive Chantal Goya ! Tu me fais un calin ?"
    print(f"   pushline : {pushline}...")
    
    return {
        "pushline": pushline,
        "tentatives": tentative,
        "historique": state.get("historique", []) + [pushline]
    }


# NŒUD 2 : Évaluer la pushline
def evaluer_pushline(state: Etatpushline) -> dict:
    """Évalue la pushline avec un score de 1 à 10"""
    
    print(f"   Évaluation de la pushline...")
    
    prompt = ChatPromptTemplate.from_template(
        """Évalue cette punchline sur 10 selon sa fidélité au style du sergent-major Tom "Gunny" Highway (Clint Eastwood) dans le film "Le Maître de Guerre".

CRITÈRES D'ÉVALUATION :
- Est-ce bourru, direct et sans filtre ? 
- Y a-t-il du sarcasme militaire tranchant ?
- Le vocabulaire est-il cru et imagé (insultes poétiques, comparaisons crues) ?
- Sent-on le vétéran endurci qui a tout vu ?
- La réplique pourrait-elle sortir de la bouche de Clint Eastwood ?

Punchline à évaluer : {pushline}

Note : 1 = pas du tout Tom Highway / 10 = parfaitement dans le personnage
Réponds UNIQUEMENT avec un nombre entre 1 et 10, rien d'autre."""
    )
    
    chain = prompt | llm_eval | StrOutputParser()
    
    try:
        score = int(chain.invoke({"pushline": state["pushline"]}).strip())
        score = max(1, min(10, score))  # Clamp entre 1 et 10
    except:
        score = 5  # Score par défaut si erreur
    
    print(f"   Score : {score}/10")
    
    return {"score": score}


### Étape 3 : Définir la Condition de Routage

C'est là que LangGraph brille ! On peut créer des **boucles conditionnelles**.


In [62]:
# Condition : continuer ou terminer ?
def decide_continuer(state: Etatpushline) -> Literal["generer", "fin"]:
    """Décide si on réessaie ou si on termine"""
    
    # Maximum 3 tentatives pour éviter les boucles infinies
    if state["tentatives"] >= 3:
        print("   Maximum de tentatives atteint !")
        return "fin"
    
    # Si le score est >= 8, on est satisfait
    if state["score"] >= 8:
        print("   pushline validée !")
        return "fin"
    else:
        print("   Score insuffisant, on réessaie...")
        return "generer"

print("Condition définie : score >= 7 ou max 3 tentatives")


Condition définie : score >= 7 ou max 3 tentatives


### Étape 4 : Construire le Graphe

On assemble les nœuds et les arêtes (transitions).


In [63]:
# On construit le graphe
graph = StateGraph(Etatpushline)

# On ajoute les nœuds
graph.add_node("generer", generer_pushline)
graph.add_node("evaluer", evaluer_pushline)

# On ajoute les arêtes (ce sont les transitions)
graph.add_edge(START, "generer")        # Départ → Générer
graph.add_edge("generer", "evaluer")    # Générer → Évaluer

# On fait une arête conditionnelle : après évaluation, on décide
graph.add_conditional_edges(
    "evaluer",                           # Depuis le nœud "evaluer"
    decide_continuer,                    # Fonction de décision
    {
        "generer": "generer",            # Si "generer" → retour au nœud generer
        "fin": END                        # Si "fin" → terminer
    }
)

# Compiler le graphe
app = graph.compile()

---
## Test du Graphe LangGraph

Observez les **boucles** : si le score est < 7, le graphe **réessaie automatiquement** !


In [66]:
# Exécuter le graphe !
theme_test = "Intelligence artificielle"

print("=" * 60)
print(f"THÈME : {theme_test}")
print("=" * 60)

resultat = app.invoke({
    "theme": theme_test,
    "pushline": "",
    "score": 0,
    "tentatives": 0,
    "historique": []
})

print("\n" + "=" * 60)
print("RÉSULTAT FINAL :")
print("=" * 60)
print(f"pushline finale : {resultat['pushline']}")
print(f"Score : {resultat['score']}/10")
print(f"Tentatives : {resultat['tentatives']}")


THÈME : Intelligence artificielle

Tentative 1 : Génération d'une pushline du sergent-major Highway...
   pushline : "T'es en train de me dire que tu veux confier ta peau à une bande de 1 et de 0 ? L'intelligence artificielle, c'est juste une pute de Silicon Valley qui te promet du rêve mais te baissera dès que les balles commenceront à voler. Tu veux miser ta vie sur un truc qui peut se crasher comme un bleu sous tir d'artillerie ? Faut être un sacré trou du cul pour croire ça."...
   Évaluation de la pushline...
   Score : 8/10
   pushline validée !

RÉSULTAT FINAL :
pushline finale : "T'es en train de me dire que tu veux confier ta peau à une bande de 1 et de 0 ? L'intelligence artificielle, c'est juste une pute de Silicon Valley qui te promet du rêve mais te baissera dès que les balles commenceront à voler. Tu veux miser ta vie sur un truc qui peut se crasher comme un bleu sous tir d'artillerie ? Faut être un sacré trou du cul pour croire ça."
Score : 8/10
Tentatives : 1


---
## Résumé : LangChain vs LangGraph

| Aspect | LangChain | LangGraph |
|--------|-----------|-----------|
| **Structure** | Chaîne linéaire (A→B→C) | Graphe (nœuds + arêtes) |
| **Boucles** | ❌ Impossible | ✅ Oui ! |
| **Conditions** | Limitées | ✅ Branchements conditionnels |
| **État** | Passe de maillon en maillon | État global persistant |
| **Cas d'usage** | Pipelines simples | Agents, workflows complexes |

### Quand utiliser quoi ?

- **LangChain** : RAG simple, chaînes de prompts, transformations linéaires
- **LangGraph** : Agents avec réflexion, workflows avec validation, boucles de feedback
